# doc-extract-lora: baseline + LoRA fine-tuning

Fine-tunes a small open LLM to extract structured JSON from invoice/receipt text, and benchmarks it against the un-fine-tuned base model.

**Before running:** `Runtime > Change runtime type > T4 GPU` (free tier is enough for a 1.5B-3B model with QLoRA).

This notebook is self-contained - it re-fetches and re-derives the dataset from Hugging Face rather than depending on the local repo, so it runs standalone in Colab. The logic here mirrors `data/schema.py`, `scripts/fetch_dataset.py`, `scripts/convert_invoices.py`, `data/prepare.py`, `training/train_lora.py`, and `eval/metrics.py` in the [doc-extract-lora](.) repo - keep both in sync if you change the task/schema.

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets jsonschema

## 1. Fetch the dataset

Pulls all rows of `mychen76/invoices-and-receipts_ocr_v1` via the HF datasets-server REST API (text fields only - the parquet loader bundles full images and is too slow/large for this task).

In [ ]:
import json, time, urllib.request

DATASET = "mychen76/invoices-and-receipts_ocr_v1"
API = "https://datasets-server.huggingface.co/rows"
PAGE_SIZE = 100


def fetch_page(offset, length):
    url = (f"{API}?dataset={DATASET.replace('/', '%2F')}"
           f"&config=default&split=train&offset={offset}&length={length}")
    max_attempts = 8
    for attempt in range(max_attempts):
        try:
            with urllib.request.urlopen(url, timeout=30) as resp:
                return json.loads(resp.read())
        except urllib.error.HTTPError as e:
            if attempt == max_attempts - 1:
                raise
            # the API rate-limits bursts of requests (e.g. re-running this
            # cell right after a previous run) - a 429 needs a longer
            # backoff than a transient 5xx, or it just gets rate-limited again
            wait = 30 if e.code == 429 else 2 ** attempt
            print(f"  retry {attempt + 1} after HTTP {e.code} (waiting {wait}s)")
            time.sleep(wait)
        except Exception as e:
            if attempt == max_attempts - 1:
                raise
            time.sleep(2 ** attempt)


first = fetch_page(0, 1)
total = first["num_rows_total"]
print(f"dataset has {total} rows")

raw_rows = []
offset = 0
while offset < total:
    length = min(PAGE_SIZE, total - offset)
    page = fetch_page(offset, length)
    for row in page["rows"]:
        r = row["row"]
        raw_rows.append({"id": r["id"], "parsed_data": r["parsed_data"], "raw_data": r["raw_data"]})
    offset += length
    print(f"  fetched {offset}/{total}", end="\r")

print(f"\nfetched {len(raw_rows)} raw rows")

## 2. Schema + validator

The fixed extraction target. Field-level accuracy and JSON-validity in the eval harness are both scored against this.

In [ ]:
from jsonschema import Draft202012Validator, ValidationError

RECEIPT_SCHEMA = {
    "type": "object",
    "required": ["vendor", "date", "line_items", "subtotal", "tax", "total"],
    "additionalProperties": False,
    "properties": {
        "vendor": {"type": "string"},
        "date": {"type": "string", "pattern": r"^\d{4}-\d{2}-\d{2}$"},
        "line_items": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["description", "quantity", "unit_price", "amount"],
                "additionalProperties": False,
                "properties": {
                    "description": {"type": "string"},
                    "quantity": {"type": "number"},
                    "unit_price": {"type": "number"},
                    "amount": {"type": "number"},
                },
            },
        },
        "subtotal": {"type": "number"},
        "tax": {"type": "number"},
        "total": {"type": "number"},
    },
}

_validator = Draft202012Validator(RECEIPT_SCHEMA)
FIELDS = ("vendor", "date", "subtotal", "tax", "total")


def validate(record):
    try:
        _validator.validate(record)
        return True, None
    except ValidationError as e:
        return False, e.message

## 3. Convert to labeled examples

Maps this dataset's `header`/`items`/`summary` shape onto our schema. Handles two data-quality quirks: European-formatted numbers (comma decimal), and ~80% of rows leaving the structured `invoice_date` blank even though the date is present in the OCR text (recovered via regex).

In [ ]:
import ast, re
from datetime import datetime

_DATE_RE = re.compile(r"\b\d{1,2}/\d{1,2}/\d{4}\b")


def parse_number(raw):
    if raw is None:
        return None
    s = re.sub(r"[^\d,.\-]", "", str(raw)).strip()
    if not s:
        return None
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def parse_date(raw):
    for fmt in ("%m/%d/%Y", "%d/%m/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(raw.strip(), fmt).strftime("%Y-%m-%d")
        except (ValueError, AttributeError):
            continue
    return None


def date_from_ocr(ocr_words):
    for w in ocr_words:
        m = _DATE_RE.search(w)
        if m:
            d = parse_date(m.group(0))
            if d is not None:
                return d
    return None


def convert_row(row):
    try:
        parsed = json.loads(row["parsed_data"])
        j = ast.literal_eval(parsed["json"])
        raw = json.loads(row["raw_data"])
        ocr_words = ast.literal_eval(raw["ocr_words"])
    except (KeyError, SyntaxError, ValueError, json.JSONDecodeError):
        return None

    header = j.get("header", {})
    items = j.get("items", [])
    summary = j.get("summary", {})

    date = parse_date(header.get("invoice_date", "")) or date_from_ocr(ocr_words)
    if date is None:
        return None

    line_items = []
    for it in items:
        desc = it.get("item_desc")
        qty = parse_number(it.get("item_qty"))
        unit_price = parse_number(it.get("item_net_price"))
        amount = parse_number(it.get("item_net_worth") or it.get("total_net_worth"))
        if amount is None and qty is not None and unit_price is not None:
            amount = round(qty * unit_price, 2)
        if desc is None or qty is None or unit_price is None or amount is None:
            continue
        line_items.append({"description": desc, "quantity": qty, "unit_price": unit_price, "amount": amount})
    if not line_items:
        return None

    subtotal = parse_number(summary.get("total_net_worth"))
    tax = parse_number(summary.get("total_vat"))
    total = parse_number(summary.get("total_gross_worth"))
    if subtotal is None or tax is None or total is None:
        return None

    label = {
        "vendor": header.get("seller", "").strip(),
        "date": date,
        "line_items": line_items,
        "subtotal": subtotal,
        "tax": tax,
        "total": total,
    }
    if not label["vendor"]:
        return None

    ok, _ = validate(label)
    if not ok:
        return None

    return {"text": "\n".join(ocr_words), "label": label}


labeled = [r for r in (convert_row(row) for row in raw_rows) if r is not None]
print(f"{len(raw_rows)} raw rows -> {len(labeled)} clean labeled examples")

## 4. Build instruction-format prompts + train/val split

In [ ]:
import random

INSTRUCTION = (
    "Extract the following fields from the receipt text as a single JSON object: "
    "vendor, date (YYYY-MM-DD), line_items (list of {description, quantity, unit_price, amount}), "
    "subtotal, tax, total. Output only the JSON object, no other text."
)


def build_prompt(document_text):
    return f"{INSTRUCTION}\n\nReceipt text:\n{document_text.strip()}\n\nJSON:"


examples = [
    {"prompt": build_prompt(r["text"]), "completion": json.dumps(r["label"], ensure_ascii=False, separators=(",", ":"))}
    for r in labeled
]

random.seed(42)
random.shuffle(examples)
n_val = max(20, int(0.1 * len(examples)))
val_examples, train_examples = examples[:n_val], examples[n_val:]
print(f"train: {len(train_examples)}  val: {len(val_examples)}")

## 5. Eval metrics

Two things scored separately: JSON-validity rate (did the model even produce parseable, schema-conformant output?) and field-level precision/recall/F1 among valid outputs.

In [ ]:
import re

def try_parse_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.startswith("json"):
            text = text[4:]
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


def scalar_field_match(gold, pred):
    return {f: gold.get(f) == pred.get(f) for f in FIELDS}


def coerce_amount(value):
    # a base model with no reason to follow our schema may emit "amount" as
    # a currency-formatted string like "$2,620.00" instead of a plain number -
    # that must count as a scoring miss, not crash the eval run
    if isinstance(value, (int, float)):
        return float(value)
    digits = re.sub(r"[^0-9.\-]", "", str(value))
    try:
        return float(digits) if digits not in ("", "-", ".", "-.") else 0.0
    except ValueError:
        return 0.0


def line_items_prf(gold_items, pred_items):
    def key(item):
        return (str(item.get("description", "")).strip().lower(), round(coerce_amount(item.get("amount", 0)), 2))

    gold_keys = [key(i) for i in gold_items]
    pred_keys = [key(i) for i in pred_items]
    gold_remaining = list(gold_keys)
    tp = 0
    for k in pred_keys:
        if k in gold_remaining:
            tp += 1
            gold_remaining.remove(k)
    precision = tp / len(pred_keys) if pred_keys else (1.0 if not gold_keys else 0.0)
    recall = tp / len(gold_keys) if gold_keys else (1.0 if not pred_keys else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


def score_example(gold, raw_prediction):
    pred = try_parse_json(raw_prediction)
    if pred is None:
        return {"valid_json": False, "schema_valid": False}
    schema_ok, _ = validate(pred)
    result = {"valid_json": True, "schema_valid": schema_ok}
    result["scalar_matches"] = scalar_field_match(gold, pred)
    items = pred.get("line_items", [])
    result["line_items"] = line_items_prf(gold.get("line_items", []), items if isinstance(items, list) else [])
    return result


def aggregate(results):
    n = len(results)
    valid_json_rate = sum(r["valid_json"] for r in results) / n
    schema_valid_rate = sum(r["schema_valid"] for r in results) / n
    scored = [r for r in results if r.get("scalar_matches")]
    field_accuracy = {}
    if scored:
        for f in FIELDS:
            field_accuracy[f] = sum(r["scalar_matches"][f] for r in scored) / len(scored)
    line_item_f1 = sum(r["line_items"]["f1"] for r in scored) / len(scored) if scored else 0.0
    return {
        "n_examples": n,
        "json_validity_rate": valid_json_rate,
        "schema_validity_rate": schema_valid_rate,
        "field_accuracy": field_accuracy,
        "mean_field_accuracy": sum(field_accuracy.values()) / len(field_accuracy) if field_accuracy else 0.0,
        "line_items_f1": line_item_f1,
    }

## 6. Load the base model

Default is Qwen2.5-1.5B-Instruct, sized to fit comfortably on a free-tier T4 with 4-bit QLoRA. Swap to `Qwen/Qwen2.5-3B-Instruct` if you have a bigger GPU (Colab Pro / A100) for a stronger baseline.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# free-tier T4 is Turing (compute capability 7.5) and does NOT support bf16 -
# that needs Ampere+ (compute capability 8.0+, e.g. A100 on Colab Pro).
# Detect instead of hardcoding so this notebook works on either.
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'} | using {COMPUTE_DTYPE}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")

## 7. Baseline: zero-shot eval of the base model

In [ ]:
import time


def generate(model, tok, prompt, max_new_tokens=512):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def run_eval(model, tok, examples):
    results, latencies = [], []
    for ex in examples:
        gold = json.loads(ex["completion"])
        t0 = time.perf_counter()
        raw_output = generate(model, tok, ex["prompt"])
        latencies.append(time.perf_counter() - t0)
        results.append(score_example(gold, raw_output))
    summary = aggregate(results)
    summary["mean_latency_sec"] = sum(latencies) / len(latencies)
    return summary


baseline_summary = run_eval(base_model, tokenizer, val_examples)
print(json.dumps(baseline_summary, indent=2))

## 8. LoRA fine-tuning

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_ds = Dataset.from_list(train_examples)
val_ds = Dataset.from_list(val_examples)
# each example is already {"prompt": ..., "completion": ...} - trl recognizes
# this shape as its "prompt-completion" dataset format and automatically
# masks the loss to the completion tokens only (better than training on the
# concatenated prompt+completion text, which would waste capacity learning
# to predict the fixed instruction text). No formatting_func needed - in
# fact passing one is now a hard error when the dataset has this shape.

sft_config = SFTConfig(
    output_dir="checkpoints/doc-extract-lora",
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    max_length=1024,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=USE_BF16,
    fp16=not USE_BF16,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)
trainer.train()

ADAPTER_DIR = "checkpoints/doc-extract-lora/final_adapter"
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

## 9. Eval the fine-tuned model

In [ ]:
finetuned_summary = run_eval(model, tokenizer, val_examples)
print(json.dumps(finetuned_summary, indent=2))

## 10. Compare

In [ ]:
import pandas as pd

rows = []
for name, s in [("base (zero-shot)", baseline_summary), ("fine-tuned (LoRA)", finetuned_summary)]:
    rows.append({
        "model": name,
        "json_validity": s["json_validity_rate"],
        "schema_validity": s["schema_validity_rate"],
        "mean_field_accuracy": s["mean_field_accuracy"],
        "line_items_f1": s["line_items_f1"],
        "mean_latency_sec": s["mean_latency_sec"],
    })
pd.DataFrame(rows)

## 11. Next steps

- **Ceiling comparison:** run the same `val_examples` prompts through a large hosted model (GPT-4o-class or Claude) to see how close the small fine-tuned model gets - this is the number that makes the "small model, big model quality, fraction of the cost" story concrete for a resume/write-up.
- **Quantize for edge deployment:** export the merged model to GGUF (`llama.cpp`'s `convert_hf_to_gguf.py`) or ONNX, then run it in-browser (transformers.js/WebGPU) or as a lightweight API.
- **Save the adapter somewhere durable** (Google Drive / Hugging Face Hub) - Colab runtimes are ephemeral and `checkpoints/` will be lost when the session ends:
```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r checkpoints/doc-extract-lora/final_adapter /content/drive/MyDrive/doc-extract-lora-adapter
```
- **Wire into the demo:** copy the adapter (merged with the base model) into the path `app/webdemo/main.py` expects (`DOC_EXTRACT_MODEL_PATH`) in the main repo, then `uvicorn app.webdemo.main:app --reload`.